In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import itertools

In [ ]:
n_seeds = 10
n_iters = 1_000
env_name_arr = [
    "garnet_50", 
    "garnet_200", 
    "garnet_1000", 
]
gamma_arr = [0.9, 0.99, 0.995]

# for plotting
label_arr = ["SPMD MC-Fixed", "SPMD MC-Est", "SPMD MC-Dyn"]
color_arr = ["blue", "black", "red"]
lss_arr   = ["dotted", "dashdot", "solid"]

# Summary
Plot all final performance of SPMD-MC on Garnet and GridWorld.

## SPMD MC-SPMD on Garnet and GridWorld

- Garnet: `2026_04_08/exp_2`
- GridWorld: `2026_04_15/exp_2`

In [ ]:
exp_date = "2026_04_08"; env_name = "Garnet"
# exp_date = "2026_04_15"; env_name = "GridWorld"

In [ ]:
score_1 = np.zeros((27, n_seeds, n_iters))
sample_1 = np.zeros((27, n_seeds, n_iters))
length_1 = np.zeros((27, n_seeds), dtype=int)
root = os.path.join("..", "logs")
for i in range(len(score_1)):
    for j in range(n_seeds):
        df = pd.read_csv(os.path.join(root, exp_date, "exp_2", "run_%d" % i, "seed=%d.csv" % j))
        vals = df['point value'] # df['true_value']
        length_1[i,j] = df_len = len(vals)
        score_1[i,j,:df_len] = vals

        df = pd.read_csv(os.path.join(root, exp_date, "exp_2", "run_%d" % i, "mixing_seed=%d.csv" % j))
        sample_1[i,j,:df_len] = df['cum_samples'][:df_len]
        sample_1[i,j,:df_len] += df['cum_est_samples'][:df_len]

opt_1 = np.zeros((9, n_seeds))
for i in range(len(opt_1)):
    for j in range(n_seeds):
        df = pd.read_csv(os.path.join(root, exp_date, "exp_1", "run_%d" % i, "seed=%d.csv" % j))
        opt_1[i,j] = df['average value'].iloc[-1]

Now we will plot.

In [ ]:
i_d = 4

In [ ]:
plt.style.use('ggplot')
_, axes = plt.subplots(ncols=3, figsize=(15,4))

for i in range(i_d*3,(i_d+1)*3):
    for j in range(n_seeds):
        n = length_1[i,j]
        xs = sample_1[i,j,:n]
        ys = score_1[i,j,:n] # - opt_1[i//3,j]
        axes[i%3].plot(xs,ys)

plt.suptitle(env_name)
for i in range(3):
    axes[i].set(
        title=label_arr[i],
    )

We can only plot the different seeds separately since they have different mixing times. 
We will apply a bucketing technique to try to alleviate this.

In [ ]:
buckets = np.append(
    np.append(np.arange(1e4, step=1e2), np.arange(1e4, 1e5, step=1e3)), 
    np.append(
        np.append(np.arange(1e5,1e6, step=1e4), np.arange(1e6,1e7, step=1e5)),
        np.arange(1e7,1e8, step=1e5)
    ),
)

# Bucketed scores. 1st index: number of exp, 2nd: seeds, 3rd: buckets
clean_1 = np.zeros((len(score_1), n_seeds, len(buckets)))
# Which bucket is last
clean_len_1 = np.zeros((len(score_1), n_seeds))
last_idx = -np.inf
for i in range(clean_1.shape[0]):
    for j in range(n_seeds):
        clean_len_1[i,j] = length_1[i,j]
        for k,x in enumerate(buckets):
            # finds index i such that len_i < x <= len_(i+1)
            if length_1[i,j] == 0:
                idx = -1
            else:
                idx = np.argmax(x <= sample_1[i,j,:length_1[i,j]])
            if idx != -1:
                clean_1[i,j,k]
                score_1[i,j,idx]
                clean_1[i,j,k] = score_1[i,j,idx]
            if idx < last_idx:
                clean_len_1[i,j] = k
            last_idx = idx

In [ ]:
plt.style.use('ggplot')
_, ax = plt.subplots(figsize=(5,4))

z = 2.228

for k,i in enumerate(range(i_d*3,(i_d+1)*3)):
    least_len_idx = int(np.min(clean_len_1[i,:]))
    xs = buckets[:least_len_idx]
    mean = np.mean(clean_1[i,:,:least_len_idx] - np.outer(opt_1[i//3], np.ones(least_len_idx)), axis=0) 
    sig  = np.std(clean_1[i,:,:least_len_idx], axis=0)
    ax.plot(xs, mean, label=label_arr[k], color=color_arr[k], linestyle=lss_arr[k])
    ax.fill_between(xs, mean - z*sig, mean+z*sig, color=color_arr[k], alpha=0.05)

ax.legend()
ax.set(
    title=env_name,
    xlabel="Samples",
    ylabel=r"Average optimality gap",
    xlim=(0,8e7),
)